In [ ]:
import pandas as pd

df = pd.read_csv(
    "welt_ukraine_russia_with_metadata.csv",
    low_memory=False
)

for c in df.columns:
    print(c)

print("\n" + "=" * 80)

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (
            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print("\n" + "=" * 80)

important = [

    "url",
    "published",
    "title",
    "author",
    "text",
    "image_urls",
    "image_count",
    "category"

]

print("\nIMPORTANT FIELDS\n")

for field in important:

    if field in df.columns:

        print(f"{field:20} OK")

    else:

        print(f"{field:20} MISSING")

In [ ]:
df.head()

In [ ]:
russia_keywords = [
    "russland",
    "russisch",
    "russe",
    "russen",
    "kreml",
    "kremlin",
    "moskau",
    "putin",
    "lawrow",
    "peskow",
    "schoigu",
    "gerassimow",
    "medwedew",
    "sacharowa",
    "mischustin",
    "gazprom",
    "rosneft",
    "rosatom",
    "transneft",
    "lukoil",
    "duma",
    "staatsduma",
    "krim",
    "ukrainekrieg", 
    "angriffskrieg"
]

pattern = "|".join(russia_keywords)

russia_df = df[
    df["text"]
    .fillna("")
    .str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print(len(russia_df))

In [ ]:
# compare filtered corpus with old high recall corpus

old_df = pd.read_csv(
    "welt_ukraine_russia_high_recall.csv"
)

old_urls = set(old_df["url"])
new_urls = set(russia_df["url"])

print(f"old corpus: {len(old_urls)}")
print(f"new corpus: {len(new_urls)}")

In [ ]:
# compare overlap

print(f"overlap: {len(old_urls & new_urls)}")
print(f"only old: {len(old_urls - new_urls)}")
print(f"only new: {len(new_urls - old_urls)}")

In [ ]:
# find new articles

missing_articles = russia_df[
    ~russia_df["url"].isin(old_urls)
].copy()

print(len(missing_articles))

display(
    missing_articles[
        [
            "published",
            "title",
            "url"
        ]
    ].head()
)

In [ ]:
russia_df.to_csv(
    "welt_SECONDROUND_final_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
russia_df

### Copying files

In [ ]:
import os
import shutil
import pandas as pd

# --------------------------------------------------
# FILES
# --------------------------------------------------

CSV_FILE = "welt_SECONDROUND_final_articles.csv"

OLD_IMAGE_FOLDER = "."

NEW_IMAGE_FOLDER = "."

os.makedirs(
    NEW_IMAGE_FOLDER,
    exist_ok=True
)

# --------------------------------------------------
# LOAD
# --------------------------------------------------

df = pd.read_csv(
    CSV_FILE,
    low_memory=False
)

needed = set()

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url:
            continue

        filename = url.split("/")[-1]

        needed.add(filename)

print(f"Articles: {len(df):,}")
print(f"Unique images needed: {len(needed):,}")

In [ ]:
import os

files = os.listdir(OLD_IMAGE_FOLDER)

print("Всего файлов:", len(files))
print()

for f in files[:30]:
    print(f)

In [ ]:
import pandas as pd

articles = pd.read_csv(
    "welt_SECONDROUND_final_articles.csv",
    low_memory=False
)

print(articles["image_urls"].iloc[0])
print(type(articles["image_urls"].iloc[0]))

In [ ]:
print("rows:", len(articles))

print("filled image_urls:",
      articles["image_urls"].notna().sum())

print("missing image_urls:",
      articles["image_urls"].isna().sum())

print()

print(
    articles.loc[
        articles["image_urls"].notna(),
        "image_urls"
    ].head(5)
)

In [ ]:
import re

needed_ids = set()

for urls in articles["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        m = re.search(
            r"/([a-f0-9]{32})/",
            url
        )

        if m:
            needed_ids.add(
                m.group(1)
            )

print(len(needed_ids))

In [ ]:
import os

filename = "ada4069259460f8de2877b67f7fd5fce.jpg"

print(os.path.exists(os.path.join(".", filename)))

In [ ]:
import os
import re
import pandas as pd

# paths

ARTICLES_FILE = (
    "welt_SECONDROUND_final_articles.csv"
)

SOURCE_FOLDER = (
    "."
)

# load articles

df = pd.read_csv(
    ARTICLES_FILE,
    low_memory=False
)

# collect image ids

image_ids = set()

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url:
            continue

        match = re.search(
            r"/([a-f0-9]{32})/",
            url
        )

        if match:

            image_ids.add(
                match.group(1)
            )

print("source folder:")
print(SOURCE_FOLDER)
print()

print("folder exists:")
print(os.path.exists(SOURCE_FOLDER))
print()

print("images found:", len(image_ids))
print()

# check first image only

first_id = next(iter(image_ids))

source = os.path.join(
    SOURCE_FOLDER,
    first_id + ".jpg"
)

print("first image id:")
print(first_id)
print()

print("full path:")
print(source)
print()

print("exists:")
print(os.path.exists(source))
print()

print("absolute path:")
print(os.path.abspath(source))

In [ ]:
import os
import re
import shutil
import pandas as pd
from tqdm import tqdm

# paths

ARTICLES_FILE = (
    "welt_SECONDROUND_final_articles.csv"
)

SOURCE_FOLDER = (
    "."
)

OUTPUT_FOLDER = (
    "."
)

MISSING_FILE = (
    "welt_missing_images.csv"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# load articles

df = pd.read_csv(
    ARTICLES_FILE,
    low_memory=False
)

# collect unique image ids

image_ids = set()

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url:
            continue

        match = re.search(
            r"/([a-f0-9]{32})/",
            url
        )

        if match:

            image_ids.add(
                match.group(1)
            )

print(f"articles: {len(df):,}")
print(f"unique images needed: {len(image_ids):,}")

# copy images

copied = 0
already_exists = 0
missing = []

for image_id in tqdm(sorted(image_ids)):

    source = os.path.join(
        SOURCE_FOLDER,
        image_id + ".jpg"
    )

    destination = os.path.join(
        OUTPUT_FOLDER,
        image_id + ".jpg"
    )

    if not os.path.exists(source):

        missing.append(image_id)

        continue

    if os.path.exists(destination):

        already_exists += 1

        continue

    shutil.copy2(
        source,
        destination
    )

    copied += 1

# save missing ids

pd.DataFrame({

    "image_id": missing

}).to_csv(

    MISSING_FILE,

    index=False,

    encoding="utf-8-sig"
)

print()
print(f"copied: {copied:,}")
print(f"already existed: {already_exists:,}")
print(f"missing: {len(missing):,}")

print()
print("saved folder:")
print(OUTPUT_FOLDER)

print()
print("missing file:")
print(MISSING_FILE)

In [ ]:
import re
import pandas as pd

INPUT_FILE = (
    "welt_SECONDROUND_final_articles.csv"
)

OUTPUT_FILE = (
    "welt_SECONDROUND_final_articles.csv"
)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

filenames = []

for urls in df["image_urls"].fillna(""):

    article_files = []

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        match = re.search(
            r"/([a-f0-9]{32})/",
            url
        )

        if match:

            article_files.append(
                match.group(1) + ".jpg"
            )

    filenames.append(
        "|".join(article_files)
    )

df["image_filenames"] = filenames

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("done")

In [ ]:
df

In [ ]:
print(df[["image_filenames"]].head(10))